# TB Portals - DA-MoE **mode = `fusion`**

Gate fuses A2 + A3 views per image (flagship). To also fuse the A1 detection view, generate a det_alp CSV (see the a1 notebook) and pass `--det-alp-csv`.

**Anchor vs the locked baselines** (`baseline_runs/BASELINE_COMPARISON.md`) — Timika MAE per country:
- A2: Romania 20.11 / **Moldova 30.68** / Kazakhstan 21.35
- A3: Romania 20.26 / **Moldova 26.16** / Kazakhstan 21.90
- A1: Romania 26.84 / **Moldova 32.76** / Kazakhstan 21.87

Moldova is the target.

**Attach datasets:** `tb-portals-cxr-pngs`, `medsam-vit-b`.

## 0 - Clone the codebase  *(restart kernel after any pull that changed .py)*

In [1]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + "/scripts"):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)
# After a git pull that changed .py modules, RESTART the kernel so Python reloads them.

Cloning into '/kaggle/working/dl-project-codebase'...


repo ready at /kaggle/working/dl-project-codebase


Updating files: 100% (446/446), done.


## Install deps

In [2]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segment-anything", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.6 MB/s eta 0:00:00
deps installed


## Paths - edit dataset slugs if yours differ

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import os
path = "/kaggle/input/private-data-source/medsam_vit_b.pth"
print("Exists:", os.path.exists(path))
print("Size (bytes):", os.path.getsize(path))

In [4]:
import os
WORK           = "/kaggle/working"
REPO_DIR       = "/kaggle/working/dl-project-codebase"
DATASET        = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT  = f"{DATASET}/kaggle_export"
MEDSAM_CKPT    = "/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
LUNG_DECODER   = f"{REPO_DIR}/checkpoints/component4/component4_mask_decoder.pt"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
CROPS_DIR      = f"{WORK}/crops"
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("MEDSAM_CKPT:  ", MEDSAM_CKPT, "->", os.path.isfile(MEDSAM_CKPT))
print("LUNG_DECODER: ", LUNG_DECODER, "->", os.path.isfile(LUNG_DECODER))

KAGGLE_EXPORT: /kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs/kaggle_export -> True
MEDSAM_CKPT:   /kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth -> True
LUNG_DECODER:  /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt -> True


## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

In [5]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

country       need_c  have_c  need_n  have_n
Georgia          713    1298     701    1108
Belarus          254     285     798     894
Ukraine          500    1206     816    1775
Kazakhstan       159     304     240     384
Romania          143     233      77     171
Moldova          193     278     396     527
Azerbaijan         5       7      12      18
India              0       6       3      12
Paper manifest: 5010 images (target 5010) -> /kaggle/working/tbportals_manifest_paper.csv


## 2 - MedSAM lung crops (~25 min first time; idempotent)

In [7]:
import os, sys
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from cache_lung_crops import main as crops_main
argv = ["--manifest", PAPER_MANIFEST, "--out-dir", CROPS_DIR,
        "--medsam-ckpt", MEDSAM_CKPT, "--size", "224", "--pad", "32"]
if os.path.isfile(LUNG_DECODER): argv += ["--lung-decoder-ckpt", LUNG_DECODER]
crops_main(argv)
print("crops ->", CROPS_DIR, "| count:", len(os.listdir(CROPS_DIR)))

[crops] device=cuda:Tesla T4
[crops] 0/5010 cached; generating the remaining 5010.
[crops] loaded fine-tuned lung decoder: /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt
[crops] 200/5010 (lung=200, fallback=0)
[crops] 400/5010 (lung=400, fallback=0)
[crops] 600/5010 (lung=600, fallback=0)
[crops] 800/5010 (lung=800, fallback=0)
[crops] 1000/5010 (lung=1000, fallback=0)
[crops] 1200/5010 (lung=1200, fallback=0)
[crops] 1400/5010 (lung=1400, fallback=0)
[crops] 1600/5010 (lung=1600, fallback=0)
[crops] 1800/5010 (lung=1800, fallback=0)
[crops] 2000/5010 (lung=2000, fallback=0)
[crops] 2200/5010 (lung=2200, fallback=0)
[crops] 2400/5010 (lung=2400, fallback=0)
[crops] 2600/5010 (lung=2600, fallback=0)
[crops] 2800/5010 (lung=2800, fallback=0)
[crops] 3000/5010 (lung=3000, fallback=0)
[crops] 3200/5010 (lung=3200, fallback=0)
[crops] 3400/5010 (lung=3400, fallback=0)
[crops] 3600/5010 (lung=3600, fallback=0)
[crops] 3800/5010 (lung=3800, fallback=0)
[

## 3 - Configure

In [8]:
# ---- this notebook is dedicated to MODE = "fusion" --------------------------
MODE     = "fusion"
SEEDS    = ["0", "1", "2"]   # full run; use ["0"] first if you want a fast sanity check
EPOCHS   = "30"
PRETRAIN = "10"              # phase-1 expert-pretraining epochs (rest = gate+critic+DANN)
OUT_DIR  = f"/kaggle/working/checkpoints/moe_{MODE}"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print("will run MoE mode =", MODE, "seeds =", SEEDS, "->", OUT_DIR)

will run MoE mode = fusion seeds = ['0', '1', '2'] -> /kaggle/working/checkpoints/moe_fusion


## 4 - Train + evaluate

Per (country, seed): frozen cavity agent (if used) -> MoE (phase 1 experts -> phase 2 gate+critic+DANN). ~2-3 h for 3 seeds.

In [9]:
from src.training.train_da_moe import main as moe_main
argv = ["--mode", MODE, "--manifest", PAPER_MANIFEST, "--crops-dir", CROPS_DIR,
        "--out-dir", OUT_DIR, "--held-outs", "Romania", "Moldova", "Kazakhstan",
        "--seeds", *SEEDS, "--epochs", EPOCHS, "--pretrain-epochs", PRETRAIN,
        "--batch-size", "60", "--accum-steps", "5", "--num-workers", "2"]
# a1/a2/fusion use the cavity agent on whole images (matches the locked A2 config):
if MODE in ("a1", "a2", "fusion"):
    argv += ["--cavity-no-lung-crop"]
moe_main(argv)

[da-moe] device=cuda mode=fusion dann=True critic=True

===== DA-MoE[fusion]  Romania  seed=0 =====
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[MoE] reg_train=3832 val=958 test=220 | 7 train-countries for DANN
[MoE][CAV] train=2918 val=730 (balanced)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 149MB/s] 


  [CAV] epoch 00 train_ce=0.76160 val_ce=0.77309
  [CAV] epoch 01 train_ce=0.66519 val_ce=0.57411
  [CAV] epoch 02 train_ce=0.58317 val_ce=0.58633
  [CAV] epoch 03 train_ce=0.56414 val_ce=0.58400
  [CAV] epoch 04 train_ce=0.54477 val_ce=0.64643
  [CAV] epoch 05 train_ce=0.56450 val_ce=0.56990
  [CAV] epoch 06 train_ce=0.51555 val_ce=0.60739
  [CAV] epoch 07 train_ce=0.49798 val_ce=0.64040
  [CAV] epoch 08 train_ce=0.48311 val_ce=0.58982
  [CAV] epoch 09 train_ce=0.48005 val_ce=0.57818
  [CAV] epoch 10 train_ce=0.50256 val_ce=0.74840
  [CAV] epoch 11 train_ce=0.44859 val_ce=0.59746
  [CAV] epoch 12 train_ce=0.44446 val_ce=0.58253
  [CAV] epoch 13 train_ce=0.39945 val_ce=0.63592
  [CAV] epoch 14 train_ce=0.40683 val_ce=0.66028
  [CAV] epoch 15 train_ce=0.41557 val_ce=0.72637
  [CAV] epoch 16 train_ce=0.38960 val_ce=0.64500
  [CAV] epoch 17 train_ce=0.35581 val_ce=0.89681
  [CAV] epoch 18 train_ce=0.36810 val_ce=0.69337
  [CAV] epoch 19 train_ce=0.30026 val_ce=1.14429
  [CAV] epoch 20 tra

## 5 - Save results + trained models (download both)

In [10]:
import os, shutil
# results CSV alone (small — quick to grab)
csv_dst = f"/kaggle/working/results_moe_{MODE}.csv"
shutil.copy(f"{OUT_DIR}/results_moe.csv", csv_dst)
# trained models + results -> one zip (the moe_*.pt checkpoints live in OUT_DIR)
zip_path = shutil.make_archive(f"/kaggle/working/checkpoints_moe_{MODE}", "zip", OUT_DIR)
n_ckpt = len([f for f in os.listdir(OUT_DIR) if f.endswith(".pt")])
print("Saved:")
print("  ", csv_dst, " (results only)")
print("  ", zip_path, f" ({n_ckpt} trained .pt models + results_moe.csv)")
print("Download BOTH from the Output panel. Drop the CSV into baseline_runs/MoE/.")

Saved:
   /kaggle/working/results_moe_fusion.csv  (results only)
   /kaggle/working/checkpoints_moe_fusion.zip  (9 trained .pt models + results_moe.csv)
Download BOTH from the Output panel. Drop the CSV into baseline_runs/MoE/.


## 6 - Ablations (optional, run last)

In [11]:
# ---- ablations (run last). RESTART-SAFE: re-hydrates config from on-disk files,
# since /kaggle/working survives a kernel restart but Python variables do not.
import os, sys
try:
    MODE
except NameError:
    MODE = "fusion"   # this notebook's mode
WORK = "/kaggle/working"; REPO_DIR = f"{WORK}/dl-project-codebase"
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
PAPER_MANIFEST = globals().get("PAPER_MANIFEST", f"{WORK}/tbportals_manifest_paper.csv")
CROPS_DIR      = globals().get("CROPS_DIR", f"{WORK}/crops")
EPOCHS         = globals().get("EPOCHS", "30")
PRETRAIN       = globals().get("PRETRAIN", "10")
_det = f"{WORK}/det_alp.csv"
if MODE == "a1" and "DET_ALP_CSV" not in globals() and os.path.isfile(_det):
    DET_ALP_CSV = _det
if MODE == "a1":
    print("NOTE: a1 has no learnable experts, so --no-dann / --no-both train nothing "
          "(degenerate floor). These ablations are informative mainly for a2/a3/fusion.")

from src.training.train_da_moe import main as moe_main
def run(tag, extra):
    out = f"/kaggle/working/checkpoints/moe_{MODE}_abl_{tag}"
    os.makedirs(out, exist_ok=True)
    base = ["--mode", MODE, "--manifest", PAPER_MANIFEST, "--crops-dir", CROPS_DIR,
            "--out-dir", out, "--held-outs", "Romania", "Moldova", "Kazakhstan",
            "--seeds", "0", "--epochs", EPOCHS, "--pretrain-epochs", PRETRAIN,
            "--batch-size", "60", "--accum-steps", "5", "--num-workers", "2"]
    if MODE in ("a1", "a2", "fusion"): base += ["--cavity-no-lung-crop"]
    if "DET_ALP_CSV" in globals(): base += ["--det-alp-csv", DET_ALP_CSV]
    print("\n==== ABLATION", MODE, tag, extra, "===="); moe_main(base + extra)

run("no_dann",   ["--no-dann"])
run("no_critic", ["--no-critic"])
run("no_both",   ["--no-dann", "--no-critic"])


==== ABLATION fusion no_dann ['--no-dann'] ====
[da-moe] device=cuda mode=fusion dann=False critic=True

===== DA-MoE[fusion]  Romania  seed=0 =====
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[MoE] reg_train=3832 val=958 test=220 | 7 train-countries for DANN
[MoE][CAV] train=2918 val=730 (balanced)
  [CAV] epoch 00 train_ce=0.76160 val_ce=0.77309
  [CAV] epoch 01 train_ce=0.66519 val_ce=0.57411
  [CAV] epoch 02 train_ce=0.58317 val_ce=0.58633
  [CAV] epoch 03 train_ce=0.56414 val_ce=0.58400
  [CAV] epoch 04 train_ce=0.54477 val_ce=0.64643
  [CAV] epoch 05 train_ce=0.56450 val_ce=0.56990
  [CAV] epoch 06 train_ce=0.51555 val_ce=0.60739
  [CAV] epoch 07 train_ce=0.49798 val_ce=0.64040
  [CAV] epoch 08 train_ce=0.48311 val_ce=0.58982
  [CAV] epoch 09 train_ce=0.48005 val_ce=0.57818
  [CAV] epoch 10 train_ce=0.50256 val_ce=0.74840
  [CAV] epoch 11 train_ce=0.44859 val_ce=0.59746
  [CAV] epoch 12 train_ce=0.44446 val_ce=0.58253
  [CAV] ep